# Lie Group Visualization

Interactive visualizations for understanding Lie group operations, manifolds, and tangent spaces.

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.linalg import expm

## Adjoint Action on Tangent Planes

The adjoint operation maps tangent vectors from one point on the manifold to another. For SO(3), the adjoint of a rotation R is simply R itself: Ad_R(v) = R·v.

We'll visualize how the tangent plane transforms under rotations around the x-axis.

In [2]:
# Omega functions from filters.py - Angular velocities in the Lie algebra
def omega_func_modelA(theta: float) -> np.ndarray:
    return np.asarray([0.3 * np.sin(6 * theta) * np.cos(6 * theta), 0.3, 0.])

def omega_func_modelB(theta: float) -> np.ndarray:
    return np.asarray([0.9 * np.sin(theta)*np.cos(theta), 0., 0.])

def omega_func_modelC(theta: float) -> np.ndarray:
    return np.asarray([0.6 * np.cos(2 * theta), 0.6 * np.cos(theta)**2, 0.])

def omega_func_modelD(theta: float) -> np.ndarray:
    return np.asarray([0.2 * np.cos(3 * theta) * np.sin(theta), 0.5 * 0.9, 0.])

def omega_func_modelE(theta: float) -> np.ndarray:
    return np.asarray([0.0, 0.0, 0.0])

def omega_func_modelF(theta: float) -> np.ndarray:
    return np.asarray([0.4 * (np.cos(theta) * np.sin(theta)-np.sin(theta)**3), 0.4*np.cos(theta)**2*np.sin(-theta), 0.])

def omega_func_constant(theta: float) -> np.ndarray:
    """Constant angular velocity of 0.2 rad/s around z-axis (creates great circle on sphere)"""
    return np.asarray([0.0, 0.0, 0.2])

OMEGA_MODELS = {
    'modelA': omega_func_modelA,
    'modelB': omega_func_modelB,
    'modelC': omega_func_modelC,
    'modelD': omega_func_modelD,
    'modelE': omega_func_modelE,
    'modelF': omega_func_modelF,
    'constant': omega_func_constant,
}

def skew(v: np.ndarray) -> np.ndarray:
    """Skew-symmetric matrix for SO(3)"""
    v = v.flatten()
    return np.array([
        [0, -v[2], v[1]],
        [v[2], 0, -v[0]],
        [-v[1], v[0], 0]
    ])

def exp_SO3(omega: np.ndarray) -> np.ndarray:
    """Exponential map for SO(3) using matrix exponential"""
    return expm(skew(omega))

In [3]:
def rotation_y(theta):
    """Rotation matrix around y-axis"""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s],
                     [0, 1, 0],
                     [-s, 0, c]])

def generate_trajectory_on_sphere(omega_func, num_points=200, dt=0.05, model_name=''):
    """
    Generate trajectory on sphere using exponential map integration.
    
    Args:
        omega_func: Function that returns angular velocity omega(theta) in Lie algebra
        num_points: Number of integration steps
        dt: Time step for integration
        model_name: Name of the model for reference
    
    Returns:
        trajectory: Array of points on the sphere (num_points x 3)
    """
    trajectory = []
    R = np.eye(3)  # Start at identity
    
    for i in range(num_points):
        # Current point on sphere (use [1,0,0] so z-axis rotation creates visible circle)
        point = R @ np.array([1, 0, 0])
        trajectory.append(point)
        
        # Get angular velocity at current phase
        theta = 2 * np.pi * i / num_points
        omega = omega_func(theta)
        
        # Integrate using exponential map: R_new = R * exp(omega * dt)
        R = R @ exp_SO3(omega * dt)
    
    return np.array(trajectory)

def project_to_tangent_plane(point_3d, plane_center):
    """
    Project a 3D point onto the tangent plane at plane_center.
    For a sphere, the tangent plane at a point is perpendicular to the radial direction.
    
    Args:
        point_3d: Point to project (3,)
        plane_center: Center of tangent plane on sphere (3,)
    
    Returns:
        Projection in local 2D coordinates on the tangent plane
    """
    # Vector from plane center to point
    v = point_3d - plane_center
    
    # Component along normal (radial) direction
    normal = plane_center / np.linalg.norm(plane_center)
    radial_component = np.dot(v, normal)
    
    # Project onto tangent plane by removing radial component
    tangent_projection = v - radial_component * normal
    
    return tangent_projection

def create_adjoint_visualization(angle_x, selected_model='constant'):
    """Create visualization with adjoint-transformed tangent plane and trajectory projections"""
    
    # Create a unit sphere
    u = np.linspace(0, 2 * np.pi, 50)
    v = np.linspace(0, np.pi, 50)
    x_sphere = np.outer(np.cos(u), np.sin(v))
    y_sphere = np.outer(np.sin(u), np.sin(v))
    z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))
    
    # Original identity point on sphere (on x-axis)
    identity_point = np.array([1, 0, 0])
    
    # Rotation matrix (group element)
    R = rotation_y(angle_x)
    
    # Rotated identity point (where the new tangent plane will be)
    rotated_identity = R @ identity_point
    
    # Generate trajectory on sphere using selected omega model
    # For constant omega = 0.2 rad/s, one loop = 2π/0.2 ≈ 31.4 seconds
    # With dt = 0.1, we need ~314 steps for one complete loop
    omega_func = OMEGA_MODELS[selected_model]
    trajectory_sphere = generate_trajectory_on_sphere(omega_func, num_points=314, dt=0.1)
    
    # Apply adjoint action to trajectory: Ad_R(trajectory) = R * trajectory
    trajectory_adjoint = (R @ trajectory_sphere.T).T
    
    # Original tangent plane at identity (perpendicular to x-axis at [1,0,0])
    plane_size = 0.8
    y_plane_coords = np.linspace(-plane_size, plane_size, 10)
    z_plane_coords = np.linspace(-plane_size, plane_size, 10)
    Y_plane_orig, Z_plane_orig = np.meshgrid(y_plane_coords, z_plane_coords)
    X_plane_orig = np.ones_like(Y_plane_orig) * 1.0
    
    # Original tangent vectors at identity (in yz-plane at x=1)
    tangent_v1 = np.array([0, 0.5, 0])  # In tangent space (y-direction)
    tangent_v2 = np.array([0, 0, 0.5])  # In tangent space (z-direction)
    
    # Apply adjoint action: Ad_R(v) = R * v
    transformed_v1 = R @ tangent_v1
    transformed_v2 = R @ tangent_v2
    
    # Create transformed tangent plane
    # Build plane perpendicular to rotated_identity at that point
    # Tangent plane is spanned by two orthogonal vectors perpendicular to rotated_identity
    
    # Get two orthogonal vectors in the tangent space at rotated_identity
    if abs(rotated_identity[0]) < 0.99:
        basis1 = np.cross(rotated_identity, np.array([1, 0, 0]))
    else:
        basis1 = np.cross(rotated_identity, np.array([0, 1, 0]))
    basis1 = basis1 / np.linalg.norm(basis1)
    basis2 = np.cross(rotated_identity, basis1)
    basis2 = basis2 / np.linalg.norm(basis2)
    
    # Create transformed tangent plane points
    plane_points = []
    for yi in y_plane_coords:
        for zi in z_plane_coords:
            point = rotated_identity + yi * basis1 + zi * basis2
            plane_points.append(point)
    plane_points = np.array(plane_points)
    X_plane_trans = plane_points[:, 0].reshape(len(y_plane_coords), len(z_plane_coords))
    Y_plane_trans = plane_points[:, 1].reshape(len(y_plane_coords), len(z_plane_coords))
    Z_plane_trans = plane_points[:, 2].reshape(len(y_plane_coords), len(z_plane_coords))
    
    # Project trajectories onto tangent planes
    traj_proj_identity = []
    traj_proj_rotated = []
    
    for point in trajectory_sphere:
        proj_id = project_to_tangent_plane(point, identity_point)
        traj_proj_identity.append(identity_point + proj_id)
    
    for point in trajectory_adjoint:
        proj_rot = project_to_tangent_plane(point, rotated_identity)
        traj_proj_rotated.append(rotated_identity + proj_rot)
    
    traj_proj_identity = np.array(traj_proj_identity)
    traj_proj_rotated = np.array(traj_proj_rotated)
    
    # Create the 3D plot
    fig = go.Figure()
    
    # Add sphere
    fig.add_trace(go.Surface(
        x=x_sphere, y=y_sphere, z=z_sphere,
        colorscale='Blues',
        opacity=0.3,
        name='Manifold (S²)',
        showscale=False
    ))
    
    # Add original tangent plane
    fig.add_trace(go.Surface(
        x=X_plane_orig, y=Y_plane_orig, z=Z_plane_orig,
        colorscale='Greens',
        opacity=0.3,
        name='Original Tangent Space',
        showscale=False
    ))
    
    # Add transformed tangent plane
    fig.add_trace(go.Surface(
        x=X_plane_trans, y=Y_plane_trans, z=Z_plane_trans,
        colorscale='Reds',
        opacity=0.3,
        name='Adjoint-Transformed Tangent Space',
        showscale=False
    ))
    
    # Add trajectory on sphere
    fig.add_trace(go.Scatter3d(
        x=trajectory_sphere[:, 0],
        y=trajectory_sphere[:, 1],
        z=trajectory_sphere[:, 2],
        mode='lines',
        line=dict(color='purple', width=6),
        name=f'Trajectory on Sphere ({selected_model})'
    ))
    
    # Add adjoint trajectory on sphere
    fig.add_trace(go.Scatter3d(
        x=trajectory_adjoint[:, 0],
        y=trajectory_adjoint[:, 1],
        z=trajectory_adjoint[:, 2],
        mode='lines',
        line=dict(color='orange', width=6),
        name=f'Ad(R) Trajectory on Sphere'
    ))
    
    # Add projected trajectory on identity tangent plane
    fig.add_trace(go.Scatter3d(
        x=traj_proj_identity[:, 0],
        y=traj_proj_identity[:, 1],
        z=traj_proj_identity[:, 2],
        mode='lines',
        line=dict(color='darkgreen', width=4, dash='dash'),
        name='Projection on Identity Plane'
    ))
    
    # Add projected trajectory on rotated tangent plane
    fig.add_trace(go.Scatter3d(
        x=traj_proj_rotated[:, 0],
        y=traj_proj_rotated[:, 1],
        z=traj_proj_rotated[:, 2],
        mode='lines',
        line=dict(color='darkred', width=4, dash='dash'),
        name='Projection on Rotated Plane'
    ))
    
    # Add original identity point
    fig.add_trace(go.Scatter3d(
        x=[identity_point[0]],
        y=[identity_point[1]],
        z=[identity_point[2]],
        mode='markers',
        marker=dict(size=8, color='green'),
        name='Original Identity'
    ))
    
    # Add rotated identity point
    fig.add_trace(go.Scatter3d(
        x=[rotated_identity[0]],
        y=[rotated_identity[1]],
        z=[rotated_identity[2]],
        mode='markers',
        marker=dict(size=8, color='red'),
        name='Rotated Identity'
    ))
    
    # Add original tangent vectors
    fig.add_trace(go.Scatter3d(
        x=[identity_point[0], identity_point[0] + tangent_v1[0]],
        y=[identity_point[1], identity_point[1] + tangent_v1[1]],
        z=[identity_point[2], identity_point[2] + tangent_v1[2]],
        mode='lines+markers',
        line=dict(color='darkgreen', width=3),
        marker=dict(size=[0, 5]),
        name='Original Tangent v₁'
    ))
    
    fig.add_trace(go.Scatter3d(
        x=[identity_point[0], identity_point[0] + tangent_v2[0]],
        y=[identity_point[1], identity_point[1] + tangent_v2[1]],
        z=[identity_point[2], identity_point[2] + tangent_v2[2]],
        mode='lines+markers',
        line=dict(color='darkgreen', width=3),
        marker=dict(size=[0, 5]),
        name='Original Tangent v₂'
    ))
    
    # Add transformed tangent vectors
    fig.add_trace(go.Scatter3d(
        x=[rotated_identity[0], rotated_identity[0] + transformed_v1[0]],
        y=[rotated_identity[1], rotated_identity[1] + transformed_v1[1]],
        z=[rotated_identity[2], rotated_identity[2] + transformed_v1[2]],
        mode='lines+markers',
        line=dict(color='darkred', width=3),
        marker=dict(size=[0, 5]),
        name='Ad(R) v₁'
    ))
    
    fig.add_trace(go.Scatter3d(
        x=[rotated_identity[0], rotated_identity[0] + transformed_v2[0]],
        y=[rotated_identity[1], rotated_identity[1] + transformed_v2[1]],
        z=[rotated_identity[2], rotated_identity[2] + transformed_v2[2]],
        mode='lines+markers',
        line=dict(color='darkred', width=3),
        marker=dict(size=[0, 5]),
        name='Ad(R) v₂'
    ))
    
    # Update layout
    fig.update_layout(
        title=f'Adjoint Action with {selected_model} Trajectory: θ = {np.degrees(angle_x):.1f}°',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            xaxis=dict(range=[-1.5, 1.5]),
            yaxis=dict(range=[-1.5, 1.5]),
            zaxis=dict(range=[-1.5, 1.5]),
            aspectmode='cube',
            camera=dict(
                eye=dict(x=2.0, y=2.0, z=1.5)
            )
        ),
        width=1100,
        height=900
    )
    
    return fig

# Create interactive visualization with slider
selected_model = 'constant'  # Using constant omega = 0.2 rad/s
angles = np.linspace(-np.pi, np.pi, 50)
frames = []

# Create frames for animation/slider
for angle in angles:
    # Recreate all data for this angle
    u = np.linspace(0, 2 * np.pi, 50)
    v = np.linspace(0, np.pi, 50)
    x_sphere = np.outer(np.cos(u), np.sin(v))
    y_sphere = np.outer(np.sin(u), np.sin(v))
    z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))
    
    identity_point = np.array([1, 0, 0])
    R = rotation_y(angle)
    rotated_identity = R @ identity_point
    
    # Generate trajectory
    omega_func = OMEGA_MODELS[selected_model]
    trajectory_sphere = generate_trajectory_on_sphere(omega_func, num_points=314, dt=0.1)
    trajectory_adjoint = (R @ trajectory_sphere.T).T
    
    plane_size = 0.8
    y_plane_coords = np.linspace(-plane_size, plane_size, 10)
    z_plane_coords = np.linspace(-plane_size, plane_size, 10)
    Y_plane_orig, Z_plane_orig = np.meshgrid(y_plane_coords, z_plane_coords)
    X_plane_orig = np.ones_like(Y_plane_orig) * 1.0
    
    tangent_v1 = np.array([0, 0.5, 0])
    tangent_v2 = np.array([0, 0, 0.5])
    transformed_v1 = R @ tangent_v1
    transformed_v2 = R @ tangent_v2
    
    if abs(rotated_identity[0]) < 0.99:
        basis1 = np.cross(rotated_identity, np.array([1, 0, 0]))
    else:
        basis1 = np.cross(rotated_identity, np.array([0, 1, 0]))
    basis1 = basis1 / np.linalg.norm(basis1)
    basis2 = np.cross(rotated_identity, basis1)
    basis2 = basis2 / np.linalg.norm(basis2)
    
    plane_points = []
    for yi in y_plane_coords:
        for zi in z_plane_coords:
            point = rotated_identity + yi * basis1 + zi * basis2
            plane_points.append(point)
    plane_points = np.array(plane_points)
    X_plane_trans = plane_points[:, 0].reshape(len(y_plane_coords), len(z_plane_coords))
    Y_plane_trans = plane_points[:, 1].reshape(len(y_plane_coords), len(z_plane_coords))
    Z_plane_trans = plane_points[:, 2].reshape(len(y_plane_coords), len(z_plane_coords))
    
    # Project trajectories
    traj_proj_identity = []
    traj_proj_rotated = []
    
    for point in trajectory_sphere:
        proj_id = project_to_tangent_plane(point, identity_point)
        traj_proj_identity.append(identity_point + proj_id)
    
    for point in trajectory_adjoint:
        proj_rot = project_to_tangent_plane(point, rotated_identity)
        traj_proj_rotated.append(rotated_identity + proj_rot)
    
    traj_proj_identity = np.array(traj_proj_identity)
    traj_proj_rotated = np.array(traj_proj_rotated)
    
    frames.append(go.Frame(
        data=[
            go.Surface(x=x_sphere, y=y_sphere, z=z_sphere, colorscale='Blues', opacity=0.3, showscale=False),
            go.Surface(x=X_plane_orig, y=Y_plane_orig, z=Z_plane_orig, colorscale='Greens', opacity=0.3, showscale=False),
            go.Surface(x=X_plane_trans, y=Y_plane_trans, z=Z_plane_trans, colorscale='Reds', opacity=0.3, showscale=False),
            go.Scatter3d(x=trajectory_sphere[:, 0], y=trajectory_sphere[:, 1], z=trajectory_sphere[:, 2],
                        mode='lines', line=dict(color='purple', width=6)),
            go.Scatter3d(x=trajectory_adjoint[:, 0], y=trajectory_adjoint[:, 1], z=trajectory_adjoint[:, 2],
                        mode='lines', line=dict(color='orange', width=6)),
            go.Scatter3d(x=traj_proj_identity[:, 0], y=traj_proj_identity[:, 1], z=traj_proj_identity[:, 2],
                        mode='lines', line=dict(color='darkgreen', width=4, dash='dash')),
            go.Scatter3d(x=traj_proj_rotated[:, 0], y=traj_proj_rotated[:, 1], z=traj_proj_rotated[:, 2],
                        mode='lines', line=dict(color='darkred', width=4, dash='dash')),
            go.Scatter3d(x=[identity_point[0]], y=[identity_point[1]], z=[identity_point[2]], 
                        mode='markers', marker=dict(size=8, color='green')),
            go.Scatter3d(x=[rotated_identity[0]], y=[rotated_identity[1]], z=[rotated_identity[2]], 
                        mode='markers', marker=dict(size=8, color='red')),
            go.Scatter3d(x=[identity_point[0], identity_point[0] + tangent_v1[0]], 
                        y=[identity_point[1], identity_point[1] + tangent_v1[1]], 
                        z=[identity_point[2], identity_point[2] + tangent_v1[2]],
                        mode='lines+markers', line=dict(color='darkgreen', width=3), marker=dict(size=[0, 5])),
            go.Scatter3d(x=[identity_point[0], identity_point[0] + tangent_v2[0]], 
                        y=[identity_point[1], identity_point[1] + tangent_v2[1]], 
                        z=[identity_point[2], identity_point[2] + tangent_v2[2]],
                        mode='lines+markers', line=dict(color='darkgreen', width=3), marker=dict(size=[0, 5])),
            go.Scatter3d(x=[rotated_identity[0], rotated_identity[0] + transformed_v1[0]], 
                        y=[rotated_identity[1], rotated_identity[1] + transformed_v1[1]], 
                        z=[rotated_identity[2], rotated_identity[2] + transformed_v1[2]],
                        mode='lines+markers', line=dict(color='darkred', width=3), marker=dict(size=[0, 5])),
            go.Scatter3d(x=[rotated_identity[0], rotated_identity[0] + transformed_v2[0]], 
                        y=[rotated_identity[1], rotated_identity[1] + transformed_v2[1]], 
                        z=[rotated_identity[2], rotated_identity[2] + transformed_v2[2]],
                        mode='lines+markers', line=dict(color='darkred', width=3), marker=dict(size=[0, 5])),
        ],
        layout=go.Layout(title_text=f'Adjoint Action with {selected_model} Trajectory: θ = {np.degrees(angle):.1f}°'),
        name=str(int(np.degrees(angle)))
    ))

# Create initial figure
initial_fig = create_adjoint_visualization(0, selected_model)

# Add frames to figure
initial_fig.frames = frames

# Add slider
sliders = [dict(
    active=25,  # Start at 0 degrees (middle of -180 to 180 range)
    yanchor="top",
    y=0,
    xanchor="left",
    x=0.1,
    currentvalue=dict(
        prefix="Rotation angle (Y-axis): ",
        visible=True,
        xanchor="right"
    ),
    pad=dict(b=10, t=50),
    len=0.9,
    steps=[dict(
        args=[[f.name], dict(
            frame=dict(duration=0, redraw=True),
            mode="immediate",
            transition=dict(duration=0)
        )],
        label=f"{int(float(f.name))}°",
        method="animate"
    ) for f in frames]
)]

initial_fig.update_layout(sliders=sliders)
initial_fig.show()

### Visualization Notes

- **Green plane**: Original tangent space at the identity (north pole)
- **Red plane**: Adjoint-transformed tangent space at the rotated point
- **Green point**: Original identity element
- **Red point**: Rotated identity element (R applied to identity)
- **Dark green arrows**: Original tangent vectors at identity
- **Dark red arrows**: Transformed tangent vectors (Ad_R applied to original vectors)
- **Purple curve**: Trajectory on sphere generated by integrating ω = [0, 0, 0.2] rad/s (rotation around z-axis) using the exponential map (one complete loop)
- **Orange curve**: Adjoint-transformed trajectory (Ad_R applied to purple trajectory)
- **Dashed green curve**: Projection of purple trajectory onto identity tangent plane (appears as a circle)
- **Dashed red curve**: Projection of orange trajectory onto rotated tangent plane

Use the slider to rotate around the Y-axis and observe how:
1. The adjoint action transforms the tangent plane and vectors
2. The trajectory on the sphere (a great circle) is transformed by Ad_R
3. The projections onto both tangent planes change - the circle projection becomes an ellipse when the plane is tilted

**Model Note**: Currently using constant ω = [0, 0, 0.2] rad/s (rotation around z-axis), which produces a great circle on the sphere in the xy-plane (one complete loop in ~31.4 seconds). Try changing `selected_model` to modelA-F to see different trajectory patterns.